
## Traditional Machine Learning vs Deep Learning

This section compares three traditional machine learning models with six deep learning experiments on the ECG arrhythmia classification task.

The traditional ML models include:
- Dummy Classifier
- Logistic Regression
- Random Forest

The deep learning models include:
- 1D CNN (Plain and Weighted)
- Bidirectional LSTM (Plain and Weighted)
- Bidirectional GRU (Plain and Weighted)

All models are evaluated using the validation dataset. The comparison considers Accuracy, Macro Precision, Macro Recall, Macro F1-score, Balanced Accuracy, and Weighted F1-score.

Macro-averaged metrics are particularly important because the dataset is highly imbalanced. They assign equal importance to each class, preventing the majority Normal class from dominating the evaluation.



## Combined Model Performance

The combined comparison includes all nine experiments, allowing traditional machine learning and deep learning approaches to be evaluated using the same set of performance metrics.

The models are primarily compared using Macro F1-score because it provides a balanced measure of precision and recall across all five ECG classes.

### Key observations

- **LSTM Plain** achieved the highest Macro F1-score of 94.43%, making it the strongest model according to this metric.
- **GRU Plain** achieved the highest accuracy of 99.02% and a Macro F1-score of 94.34%.
- **CNN Plain** achieved a Macro F1-score of 91.78%.
- **Random Forest** performed strongly among traditional ML approaches, achieving 97.91% accuracy and 90.37% Macro F1-score.
- Logistic Regression achieved a Macro F1-score of 48.07%, while the Dummy Classifier achieved only 18.12%.

These results demonstrate that traditional machine learning, particularly Random Forest, can provide a competitive baseline for ECG classification. However, the plain LSTM and GRU models achieved better overall Macro F1-scores.


In [1]:
import pandas as pd
from pathlib import Path
import os

In [ ]:
train_df = pd.read_csv(
    "../data/raw/mitbih_train.csv",
    header=None
)

X_train = train_df.iloc[:, :-1].values
y_train = train_df.iloc[:, -1].values

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (87554, 187)
y_train shape: (87554,)


In [3]:
metrics_path = "../results/metrics"

models = [
    "cnn_plain",
    "cnn_weighted",
    "lstm_plain",
    "lstm_weighted",
    "gru_plain",
    "gru_weighted"
]

reports = {}

for model in models:
    file_path = os.path.join(
        metrics_path,
        f"{model}_classification_report.csv"
    )

    reports[model] = pd.read_csv(file_path, index_col=0)

# Load training histories
histories = {}

for model in models:
    file_path = os.path.join(
        metrics_path,
        f"{model}_history.csv"
    )

    histories[model] = pd.read_csv(file_path)

print("All classification reports and histories loaded!")


All classification reports and histories loaded!


In [4]:
comparison_data = []

for model, report in reports.items():

    comparison_data.append({
        "Model": model.upper(),
        "Accuracy": report.loc["accuracy", "precision"],
        "Macro Precision": report.loc["macro avg", "precision"],
        "Macro Recall": report.loc["macro avg", "recall"],
        "Macro F1": report.loc["macro avg", "f1-score"],
        "Weighted F1": report.loc["weighted avg", "f1-score"]
    })

comparison_df = pd.DataFrame(comparison_data)

comparison_df = comparison_df.sort_values(
    by="Macro F1",
    ascending=False
).reset_index(drop=True)

comparison_df


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
0,LSTM_PLAIN,0.989949,0.964751,0.926155,0.944300,0.989744
1,GRU_PLAIN,0.990177,0.954036,0.933491,0.943392,0.990076
2,CNN_PLAIN,0.984771,0.952823,0.888935,0.917774,0.984170
3,GRU_WEIGHTED,0.971446,0.822207,0.935917,0.869326,0.973210
4,LSTM_WEIGHTED,0.970532,0.822880,0.925933,0.866027,0.972003
5,CNN_WEIGHTED,0.953552,0.768344,0.912247,0.821158,0.958495


In [5]:
metric_columns = [
    "Accuracy",
    "Macro Precision",
    "Macro Recall",
    "Macro F1",
    "Weighted F1"
]

comparison_df[metric_columns] = (
    comparison_df[metric_columns] * 100
)

comparison_df.round(2)


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
0,LSTM_PLAIN,98.99,96.48,92.62,94.43,98.97
1,GRU_PLAIN,99.02,95.40,93.35,94.34,99.01
2,CNN_PLAIN,98.48,95.28,88.89,91.78,98.42
3,GRU_WEIGHTED,97.14,82.22,93.59,86.93,97.32
4,LSTM_WEIGHTED,97.05,82.29,92.59,86.60,97.20
5,CNN_WEIGHTED,95.36,76.83,91.22,82.12,95.85


In [6]:
metrics_path = Path("../results/metrics")

baseline_df = pd.read_csv(metrics_path / "baselines.csv")

display(baseline_df)
print(baseline_df.columns.tolist())

,Model,Accuracy,Macro F1,Balanced Accuracy
0,Dummy Classifier,0.827762,0.181153,0.200000
1,Logistic Regression,0.673418,0.480704,0.762233
2,Random Forest,0.979137,0.903678,0.875772


['Model', 'Accuracy', 'Macro F1', 'Balanced Accuracy']


In [7]:
for file in sorted(metrics_path.glob("per_class_*.csv")):
    print(file.name)

per_class_dummy.csv
per_class_logistic_regression.csv
per_class_random_forest.csv


In [8]:
import numpy as np
import pandas as pd

train_idx = np.load("../data/processed/train_idx.npy")
val_idx = np.load("../data/processed/val_idx.npy")

print("Train samples:", len(train_idx))
print("Validation samples:", len(val_idx))

print("\nValidation class distribution:")
print(pd.Series(y_train[val_idx]).value_counts().sort_index())

Train samples: 74420
Validation samples: 13133

Validation class distribution:
0.0    10871
1.0      333
2.0      868
3.0       97
4.0      964
Name: count, dtype: int64


In [9]:
for model in [
    "dummy",
    "logistic_regression",
    "random_forest"
]:
    print(f"\n{'='*15} {model.upper()} {'='*15}")

    report = pd.read_csv(
        metrics_path / f"per_class_{model}.csv",
        index_col=0
    )

    display(report)


=============== DUMMY ===============


,precision,recall,f1-score,support
N,0.827762,1.000000,0.905766,10871.000000
S,0.000000,0.000000,0.000000,333.000000
V,0.000000,0.000000,0.000000,868.000000
F,0.000000,0.000000,0.000000,97.000000
Q,0.000000,0.000000,0.000000,964.000000
accuracy,0.827762,0.827762,0.827762,0.827762
macro avg,0.165552,0.200000,0.181153,13133.000000
weighted avg,0.685190,0.827762,0.749759,13133.000000



=============== LOGISTIC_REGRESSION ===============


,precision,recall,f1-score,support
N,0.969613,0.645755,0.775219,10871.000000
S,0.151890,0.699700,0.249598,333.000000
V,0.295120,0.738479,0.421711,868.000000
F,0.079286,0.824742,0.144665,97.000000
Q,0.738540,0.902490,0.812325,964.000000
accuracy,0.673418,0.673418,0.673418,0.673418
macro avg,0.446890,0.762233,0.480704,13133.000000
weighted avg,0.880762,0.673418,0.736594,13133.000000



=============== RANDOM_FOREST ===============


,precision,recall,f1-score,support
N,0.984044,0.992825,0.988415,10871.000000
S,0.919118,0.750751,0.826446,333.000000
V,0.940904,0.935484,0.938186,868.000000
F,0.855422,0.731959,0.788889,97.000000
Q,0.985216,0.967842,0.976452,964.000000
accuracy,0.979137,0.979137,0.979137,0.979137
macro avg,0.936941,0.875772,0.903678,13133.000000
weighted avg,0.978683,0.979137,0.978637,13133.000000


In [10]:
baseline_df = pd.read_csv(metrics_path / "baselines.csv")

model_mapping = {
    "Dummy Classifier": "dummy",
    "Logistic Regression": "logistic_regression",
    "Random Forest": "random_forest"
}

baseline_rows = []

for model_name, filename in model_mapping.items():

    report = pd.read_csv(
        metrics_path / f"per_class_{filename}.csv",
        index_col=0
    )

    summary = baseline_df[
        baseline_df["Model"] == model_name
    ].iloc[0]

    baseline_rows.append({
        "Model": model_name,
        "Accuracy": summary["Accuracy"] * 100,
        "Macro F1": summary["Macro F1"] * 100,
        "Balanced Accuracy": summary["Balanced Accuracy"] * 100,
        "Macro Precision": report.loc["macro avg", "precision"] * 100,
        "Macro Recall": report.loc["macro avg", "recall"] * 100,
        "Weighted F1": report.loc["weighted avg", "f1-score"] * 100
    })

baseline_comparison = pd.DataFrame(baseline_rows)

# Combine with the six DL models
final_comparison = pd.concat(
    [baseline_comparison, comparison_df],
    ignore_index=True
)

final_comparison = final_comparison.sort_values(
    "Macro F1",
    ascending=False
).reset_index(drop=True)

display(final_comparison.round(2))

,Model,Accuracy,Macro F1,Balanced Accuracy,Macro Precision,Macro Recall,Weighted F1
0,LSTM_PLAIN,98.99,94.43,NaN,96.48,92.62,98.97
1,GRU_PLAIN,99.02,94.34,NaN,95.40,93.35,99.01
2,CNN_PLAIN,98.48,91.78,NaN,95.28,88.89,98.42
3,Random Forest,97.91,90.37,87.58,93.69,87.58,97.86
4,GRU_WEIGHTED,97.14,86.93,NaN,82.22,93.59,97.32
5,LSTM_WEIGHTED,97.05,86.60,NaN,82.29,92.59,97.20
6,CNN_WEIGHTED,95.36,82.12,NaN,76.83,91.22,95.85
7,Logistic Regression,67.34,48.07,76.22,44.69,76.22,73.66
8,Dummy Classifier,82.78,18.12,20.00,16.56,20.00,74.98


In [11]:
final_comparison.to_csv(
    metrics_path / "all_models_comparison.csv",
    index=False
)


## Final Model Selection

Based on the validation results, the Bidirectional LSTM Plain model was selected for final test evaluation.

The primary reasons for selecting this model are:

- Highest Macro F1-score among all nine experiments: 94.43%.
- Highest Macro Precision: 96.48%.
- Strong Macro Recall: 92.62%.
- Strong Weighted F1-score: 98.97%.
- Consistent performance across all five ECG classes.

Although GRU Plain achieved slightly higher accuracy (99.02% compared with 98.99%), LSTM Plain achieved a marginally higher Macro F1-score.

Since the dataset is imbalanced, Macro F1-score was prioritized over accuracy during final model selection.

The selected LSTM model was subsequently evaluated on the separate test dataset in the next notebook.



## Limitations and Experimental Considerations

A few considerations should be kept in mind when interpreting these results:

1. **Class imbalance:** The dataset contains substantially more Normal ECG samples than samples from minority arrhythmia classes.

2. **Validation-based selection:** The best model was selected using validation performance. The validation results should not be interpreted as final test performance.

3. **Class weighting trade-off:** Weighted loss functions improved balanced accuracy in these experiments but generally reduced Macro F1-score compared with the corresponding plain models.

4. **Dataset limitations:** The dataset is derived from the MIT-BIH heartbeat dataset. The available files do not provide patient identifiers, so patient-level separation and patient-independent generalization cannot be verified.

5. **Clinical limitations:** These models are developed for educational and experimental purposes. Their results do not establish clinical reliability or suitability for real-world diagnosis.

The final selected model is evaluated separately using the untouched test dataset.



## Conclusion

In this notebook, three traditional machine learning models and six PyTorch-based deep learning experiments were compared using a common validation evaluation framework.

The experiments demonstrated that:

- Random Forest provided a strong traditional ML baseline.
- CNN, LSTM, and GRU architectures achieved competitive classification performance.
- Plain LSTM achieved the highest Macro F1-score.
- Plain GRU achieved the highest overall accuracy.
- Weighted loss functions improved balanced recall but did not improve overall Macro F1-score in these experiments.

The Bidirectional LSTM Plain model was selected for final evaluation based on its superior Macro F1-score and strong class-wise performance.

The next notebook evaluates this selected model on the separate test dataset to assess its performance on previously unseen samples.
